In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load data
df = pd.read_parquet('../data/fhvhv_tripdata_2026-01.parquet')

In [3]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())

Shape: (20940373, 25)

Columns: ['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag', 'cbd_congestion_fee']

Data Types:
 hvfhs_license_num               object
dispatching_base_num            object
originating_base_num            object
request_datetime        datetime64[us]
on_scene_datetime       datetime64[us]
pickup_datetime         datetime64[us]
dropoff_datetime        datetime64[us]
PULocationID                     int32
DOLocationID                     int32
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64

In [4]:
# Check target variable
print("Trip time stats (seconds):")
print(df['trip_time'].describe())

print("\nTrip miles stats:")
print(df['trip_miles'].describe())

print("\nUber vs Lyft split:")
print(df['hvfhs_license_num'].value_counts())

print("\nShared rides:")
print(df['shared_request_flag'].value_counts())

Trip time stats (seconds):
count    2.094037e+07
mean     1.125222e+03
std      7.922089e+02
min      0.000000e+00
25%      5.700000e+02
50%      9.210000e+02
75%      1.453000e+03
max      4.499800e+04
Name: trip_time, dtype: float64

Trip miles stats:
count    2.094037e+07
mean     4.637687e+00
std      5.556571e+00
min      0.000000e+00
25%      1.410000e+00
50%      2.690000e+00
75%      5.730000e+00
max      1.454430e+03
Name: trip_miles, dtype: float64

Uber vs Lyft split:
hvfhs_license_num
HV0003    15245631
HV0005     5694742
Name: count, dtype: int64

Shared rides:
shared_request_flag
N    20526369
Y      414004
Name: count, dtype: int64


In [5]:
# Take a manageable sample
df = df.sample(n=200000, random_state=42).reset_index(drop=True)

print("Working with:", df.shape)

Working with: (200000, 25)


In [6]:
# Remove trips that make no sense
df = df[df['trip_time'] > 60]        # trips longer than 1 minute
df = df[df['trip_time'] < 7200]      # trips shorter than 2 hours
df = df[df['trip_miles'] > 0.1]      # trips longer than 0.1 miles
df = df[df['trip_miles'] < 50]       # remove extreme outliers

# Drop rows with missing values in important columns
df = df.dropna(subset=['trip_time', 'trip_miles', 'PULocationID', 
                        'DOLocationID', 'request_datetime', 'pickup_datetime'])

df = df.reset_index(drop=True)
print("Clean data shape:", df.shape)

Clean data shape: (199700, 25)


In [8]:
# Extract useful features from datetime
df['hour'] = df['request_datetime'].dt.hour
df['day_of_week'] = df['request_datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['wait_time'] = (df['pickup_datetime'] - df['request_datetime']).dt.total_seconds()

# Is Uber or Lyft
df['is_uber'] = (df['hvfhs_license_num'] == 'HV0003').astype(int)

# Is airport trip
df['is_airport'] = (df['airport_fee'] > 0).astype(int)

# Target variable in minutes
df['trip_duration_mins'] = df['trip_time'] / 60

print("Features added. Shape:", df.shape)
print(df[['hour', 'day_of_week', 'is_weekend', 'wait_time', 'is_uber', 'is_airport', 'trip_duration_mins']].head())

Features added. Shape: (199700, 32)
   hour  day_of_week  is_weekend  wait_time  is_uber  is_airport  \
0    15            3           0      284.0        1           1   
1    18            6           1      159.0        1           0   
2    17            6           1       96.0        1           0   
3    13            1           0      481.0        1           0   
4    21            5           1      129.0        1           0   

   trip_duration_mins  
0           51.133333  
1           16.950000  
2            6.316667  
3           20.433333  
4            4.016667  


In [9]:
# Select features for model
features = ['trip_miles', 'hour', 'day_of_week', 'is_weekend', 
            'wait_time', 'is_uber', 'is_airport', 'PULocationID', 
            'DOLocationID', 'congestion_surcharge', 'shared_request_flag']

# Convert shared_request_flag to numeric
df['shared_request_flag'] = (df['shared_request_flag'] == 'Y').astype(int)

# Final dataset
X = df[features]
y = df['trip_duration_mins']

# Check for any remaining nulls
print("Nulls in X:", X.isnull().sum().sum())
print("Nulls in y:", y.isnull().sum())
print("\nX shape:", X.shape)
print("y shape:", y.shape)

Nulls in X: 0
Nulls in y: 0

X shape: (199700, 11)
y shape: (199700,)


In [10]:
# Save cleaned dataset
df[features + ['trip_duration_mins']].to_csv('../data/cleaned_trips.csv', index=False)
print("Saved successfully!")

Saved successfully!
